In [1]:
#| default_exp probe

In [2]:
#| hide
import json
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
from fastcore.all import Path

What this machine can build and run: the interpreter, the window, and what is already running.

Every answer here is about the machine the code runs on, so every answer differs per platform. The
platform and the interpreter are parameters rather than lookups: `backend` and `shell_ready` take a
platform name, and `py_version`, `is_framework` and `framework_python` take interpreter paths. The
decisions can be exercised from anywhere.

This module imports nothing but fastcore. It never imports pywebview, py2app or py2exe, so asking
what a machine could build costs none of them.

In [3]:
#| export
from __future__ import annotations
import json, os, subprocess, sys, sysconfig, time
from fastcore.all import Path, first

In [4]:
#| export
#: pywebview's GUI name per platform. A platform absent from here has no native window.
BACKENDS = {'darwin': 'cocoa', 'win32': 'edgechromium', 'linux': 'gtk'}

#: Framework builds macOS py2app can embed, best first. Not newest first: this is the version the
#: app ships, so the order is what projects are most likely to be on. uv's and pyenv's are absent
#: on purpose — both are python-build-standalone, `PYTHONFRAMEWORK` is empty, and py2app cannot
#: embed one in a bundle.
FRAMEWORK_CANDIDATES = (
    '/Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13',
    '/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14',
    '/Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12',
    '/opt/homebrew/opt/python@3.13/bin/python3.13',
    '/opt/homebrew/opt/python@3.14/bin/python3.14',
    '/opt/homebrew/opt/python@3.12/bin/python3.12',
    '/usr/local/opt/python@3.13/bin/python3.13',
    '/usr/local/opt/python@3.14/bin/python3.14',
    '/usr/local/opt/python@3.12/bin/python3.12',
    '/usr/bin/python3',
)

A platform absent from `BACKENDS` has no native window, and `shell_ready` says so before anything
tries to open one.

`FRAMEWORK_CANDIDATES` is read on macOS only, and nothing checks the paths exist until
`framework_python` walks them. The order is preference, not recency: 3.13 before 3.14 because the
version in the list is the version the app ships.

In [5]:
#| export
def backend(platform=None):
    "pywebview's GUI name for `platform`, or None where there is no supported webview."
    return BACKENDS.get(platform or sys.platform)

def shell_ready(platform=None):
    "`(ok, why)`: whether a native window can be opened here, and what is missing if not."
    gui = backend(platform)
    if gui is None: return False, f'no native webview backend for {platform or sys.platform}'
    try: import webview  # noqa: F401
    except ImportError as e: return False, f'pywebview is not installed ({e})'
    return True, gui

`shell_ready` answers two separate questions in one pair. `ok` is False for a platform with no
backend and for a missing pywebview, and `why` names which of the two it was. When `ok` is True,
`why` is the pywebview GUI name, which is what the caller passes on as `gui`.

Importing `webview` is how it finds out. That import happens here and nowhere else in the module.

In [6]:
backend('darwin'), backend('linux'), backend('plan9')

('cocoa', 'gtk', None)

In [7]:
test_eq(shell_ready('plan9'), (False, 'no native webview backend for plan9'))
ok, why = shell_ready('darwin')
test_eq(ok, why == 'cocoa')   # the gui name when a window can be opened, what is missing when not

In [8]:
#| export
def py_version(python):
    "The `(major, minor)` of `python`, or None when it will not say."
    try: out = subprocess.run([str(python), '-c', 'import sys;print(*sys.version_info[:2])'],
                              capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.SubprocessError): return None
    try: return tuple(int(n) for n in out.stdout.split()) or None
    except ValueError: return None

def is_framework(python=None):
    "Whether `python` (default: this interpreter) is a macOS framework build py2app can use."
    if python is None: return bool(sysconfig.get_config_var('PYTHONFRAMEWORK'))
    # JSON, not two words: `PYTHONFRAMEWORK` is empty off macOS, and splitting eats the version.
    probe = ('import json,sysconfig,sys;'
             "print(json.dumps([sysconfig.get_config_var('PYTHONFRAMEWORK') or '', "
             'list(sys.version_info[:2])]))')
    try: out = subprocess.run([str(python), '-c', probe], capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.SubprocessError): return False
    if out.returncode: return False
    try: name, version = json.loads(out.stdout.strip() or 'null')
    except (ValueError, TypeError): return False
    return bool(name) and tuple(version) >= (3, 12)

def framework_python(candidates=FRAMEWORK_CANDIDATES):
    "The first framework interpreter on this machine that py2app can build against."
    return first(p for c in candidates if (p := Path(c)).exists() and is_framework(p))

`py_version` and `is_framework` ask another interpreter about itself by running it. An interpreter
that is missing, is not a Python, or refuses to start is answered rather than raised: `None` from
`py_version`, False from `is_framework`.

`is_framework` given a path applies a version floor, and a framework build below 3.12 answers False.
`is_framework` with no argument reads this process's own `sysconfig` and applies no floor, so the
two forms can disagree about this very interpreter.

`framework_python` returns the first candidate that exists and passes `is_framework`, as a `Path`.
The order of the candidate list is the whole of the preference. None means the machine has no
framework build, which is every Linux machine and most macOS ones until someone installs one.

In [9]:
#| hide
tmp = TemporaryDirectory(); tdir = Path(tmp.name)
def fake_py(name, says):
    "An executable answering a probe the way an interpreter would, saying whatever `says` says."
    p = tdir/name; p.write_text(f'#!/bin/sh\necho {says!r}\n'); p.chmod(0o755)
    return p
fw  = fake_py('framework3.13',  json.dumps(['Python', [3, 13]]))
old = fake_py('framework3.11',  json.dumps(['Python', [3, 11]]))
pbs = fake_py('standalone3.13', json.dumps(['', [3, 13]]))
mute, chatty = fake_py('mute', ''), fake_py('chatty', 'no version here')

Three interpreters that answer the probe differently: a framework build py2app can embed, one on a
Python too old, and a python-build-standalone whose `PYTHONFRAMEWORK` is empty. The last is what uv
and pyenv install.

In [10]:
is_framework(fw), is_framework(old), is_framework(pbs)

(True, False, False)

In [11]:
#| hide
test_eq(framework_python([tdir/'nothing', pbs, old, fw]), fw)   # the first usable one, in order
test_eq(framework_python([pbs, old]), None)
test_eq(framework_python([]), None)
test_eq(py_version(sys.executable), tuple(sys.version_info[:2]))
test_eq(py_version(tdir/'nothing'), None)
test_eq(py_version(chatty), None)   # it answered, but not with a version
test_eq(py_version(mute), None)     # it exited clean and said nothing at all

In [12]:
#| export
def wait_for_http(url, timeout=60, interval=.1):
    "Block until `url` answers, or `timeout` passes. True if the server came up."
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except Exception: time.sleep(interval)
    return False

`wait_for_http` polls until the server answers. Every exception from the request is a not-yet, so a
connection refused while the server is still binding its port is not a failure. False means
`timeout` passed. It never raises.

An error status is not an answer. `urlopen` raises for a 404, and that is caught with the rest, so
a URL that is served but wrong waits out the full timeout.

In [13]:
#| hide
from http.server import HTTPServer, BaseHTTPRequestHandler
from threading import Thread
class _Root(BaseHTTPRequestHandler):
    def do_GET(self): self.send_response(200 if self.path == '/' else 404); self.end_headers()
    def log_message(self, *a): pass
srv = HTTPServer(('127.0.0.1', 0), _Root); Thread(target=srv.serve_forever, daemon=True).start()
url = f'http://127.0.0.1:{srv.server_port}/'

In [14]:
test_eq(wait_for_http(url), True)
test_eq(wait_for_http(url + 'missing', timeout=.3, interval=.05), False)   # 404 is not an answer

In [15]:
#| export
def running_from(bundle, ps_output=None):
    """The pids of processes running out of `bundle`, so a build does not overwrite a live app.

    py2app writes the bundle in place, and `python313.zip` is the running app's standard library.
    Replacing it under a live process leaves every import it has not made yet reading a file that
    is no longer the one it opened: `ZipImportError: bad local file header`, raised by whatever
    happens to import next and naming nothing that would lead anyone back to a rebuild.
    """
    target = str(Path(bundle).resolve())
    if ps_output is None:
        try: ps_output = subprocess.run(['ps', '-Ao', 'pid,command'], capture_output=True,
                                        text=True, timeout=10).stdout
        except (OSError, subprocess.SubprocessError): return []
    pids = []
    for line in ps_output.splitlines()[1:]:
        pid, _, command = line.strip().partition(' ')
        # The executable, not the whole command line: this very check names the bundle in its own
        # arguments, and so does every grep, editor and shell that has the path in it.
        exe = command.split(' ', 1)[0]
        if exe.startswith(target + os.sep) and pid.isdigit() and int(pid) != os.getpid():
            pids.append(int(pid))
    return pids

`running_from` takes `ps` output as an argument, so the decision can be exercised without a live
app. Left out, it runs `ps -Ao pid,command` itself and returns `[]` when that fails.

The first line of `ps_output` is dropped as the header.

A row matches when its executable is inside the bundle directory. Two things narrow that. The
comparison uses the first field of the command rather than the whole line, because this check names
the bundle in its own arguments and so does every grep, editor and shell that has the path open. It
compares against the bundle path plus a separator, so a sibling whose name starts the same way does
not match. This process is never in the result.

In [16]:
ps = """  PID COMMAND
  101 /Apps/Demo.app/Contents/MacOS/Demo --serve
  102 /usr/bin/grep -r /Apps/Demo.app
  103 /Apps/Demo.app.old/Contents/MacOS/Demo
  104 /bin/zsh
"""
running_from('/Apps/Demo.app', ps)

[101]

In [17]:
#| hide
test_eq(running_from('/Apps/Demo.app', f'PID COMMAND\n  {os.getpid()} /Apps/Demo.app/MacOS/Demo\n'), [])
test_eq(running_from('/Apps/Demo.app', ''), [])
test_eq(running_from('/Apps/Demo.app', '  101 /Apps/Demo.app/MacOS/Demo\n'), [])   # header, dropped

In [18]:
#| hide
srv.shutdown(); tmp.cleanup()